# EfficientNet Feature Map Visualization
This notebook visualizes how an image is processed layer by layer through EfficientNetB0.
We will load an image from the face mask dataset, preprocess it, pass it through the model, and plot the feature maps (outputs) of various layers.

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.models import Model

# Set figure size for better visualization
plt.rcParams['figure.figsize'] = (12, 8)

## 1. Load and Preprocess the Input Image
We need an image to feed into the network. We'll pick one from the dataset, resize it to 224x224 (the default for EfficientNetB0), and apply the specific preprocessing function.

In [ ]:
# Get an example image path (adjust if necessary)
# Let's pick a 'with_mask' image
dataset_dir = '../face_mask_dataset/train/with_mask'
image_files = os.listdir(dataset_dir)
sample_image_path = os.path.join(dataset_dir, image_files[0])

# Load using OpenCV
img = cv2.imread(sample_image_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Resize to 224x224
img_resized = cv2.resize(img, (224, 224))

# Plot the original resized image
plt.imshow(img_resized)
plt.title("Original Resized Image (224x224)")
plt.axis('off')
plt.show()

# Preprocess for EfficientNet (expands dims to make it a batch of 1)
# Note: EfficientNet expects inputs to be float32, and preprocess_input handles scaling
img_array = np.expand_dims(img_resized, axis=0)
img_preprocessed = preprocess_input(img_array.astype(np.float32))

print("Preprocessed image shape:", img_preprocessed.shape)

## 2. Load EfficientNetB0 and Select Layers for Visualization
EfficientNet has hundreds of layers. Visualizing all of them is too much. We will extract the outputs of specific activation layers (like after certain blocks) to see how the network's understanding of the image evolves.

In [ ]:
# Load base EfficientNetB0
base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Let's see all layer names to pick some interesting ones
# for layer in base_model.layers:
#     print(layer.name)

# We will select a few key layers at different depths of the network.
# EfficientNet is built using 'blocks'. We'll pick the activation outputs of a few blocks.
layer_names = [
    'stem_activation',        # Early edge detection
    'block2a_activation',     # Low-level features
    'block3a_activation',     # Mid-level features
    'block5a_activation',     # High-level features
    'top_activation'          # Deepest features before pooling
]

outputs = [base_model.get_layer(name).output for name in layer_names]

# Create a new model that takes the same input but outputs the feature maps for our selected layers
feature_map_model = Model(inputs=base_model.input, outputs=outputs)
feature_map_model.summary()

## 3. Generate Feature Maps
Now we pass our preprocessed image through this new model to get the feature maps.

In [ ]:
# Predict to get the feature maps
feature_maps = feature_map_model.predict(img_preprocessed)

for layer_name, feature_map in zip(layer_names, feature_maps):
    print(f"Layer: {layer_name} | Shape: {feature_map.shape}")

## 4. Visualize the Feature Maps
Each layer outputs many channels (filters). We will plot the first few channels (e.g., 16 or 64) for each selected layer in a grid.

In [ ]:
def plot_feature_maps(feature_maps, layer_names, max_filters=16):
    for layer_name, fmap in zip(layer_names, feature_maps):
        # fmap shape is (1, width, height, channels)
        # We drop the batch dimension
        fmap = fmap[0]
        
        n_filters = fmap.shape[-1]
        n_filters = min(n_filters, max_filters)
        
        # Determine grid size (e.g., 4x4 for 16 filters)
        size = int(np.ceil(np.sqrt(n_filters)))
        
        fig, axes = plt.subplots(size, size, figsize=(10, 10))
        fig.suptitle(f'Feature Maps for layer: {layer_name} \nShape: {fmap.shape}', fontsize=16)
        
        for i, ax in enumerate(axes.flat):
            if i < n_filters:
                # Plot the i-th channel/filter
                ax.imshow(fmap[:, :, i], cmap='viridis')
                ax.axis('off')
            else:
                ax.axis('off')
                
        plt.tight_layout()
        plt.subplots_adjust(top=0.9)
        plt.show()

# Plot the first 16 filters of each selected layer
plot_feature_maps(feature_maps, layer_names, max_filters=16)